In [3]:
import os
import shutil
import pydicom
import warnings

# --- CONFIGURATION ---
# CHANGE THIS PATH for each hospital run!
# Example: '../Data/AMCGH/' OR '../Data/SQUARE/'
source_path = '../Data/AMCGH/' 

print(f"--- STARTING UNIVERSAL ORGANIZER ---")
print(f"Target: {os.path.abspath(source_path)}")

if not os.path.exists(source_path):
    print("Error: Source path not found!")
else:
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Found {len(patient_folders)} patients. Processing...")

    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        # 1. Create Target Folders
        ct_dir = os.path.join(patient_dir, 'CT')
        struct_dir = os.path.join(patient_dir, 'Struct')
        dose_dir = os.path.join(patient_dir, 'Dose') # Dose files stay in root usually, but we can group them
        plan_dir = os.path.join(patient_dir, 'Plan')
        
        for d in [ct_dir, struct_dir, dose_dir, plan_dir]:
            if not os.path.exists(d): os.makedirs(d)
            
        counts = {'CT': 0, 'RTSTRUCT': 0, 'RTDOSE': 0, 'RTPLAN': 0}
        
        # 2. Scan & Move Files (Recursive)
        # We use os.walk to find files even if they are already in subfolders
        # But we SKIP our new target folders to avoid infinite loops if re-running
        for root, dirs, files in os.walk(patient_dir):
            
            # Skip if we are inside our new organized folders
            if any(x in root for x in ['CT', 'Struct', 'Dose', 'Plan']):
                continue
                
            for f in files:
                full_path = os.path.join(root, f)
                
                try:
                    # Force read header
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    mod = dcm.get("Modality", "Unknown").upper()
                    
                    target_folder = None
                    
                    if mod == 'CT':
                        target_folder = ct_dir
                        counts['CT'] += 1
                    elif mod == 'RTSTRUCT':
                        target_folder = struct_dir
                        counts['RTSTRUCT'] += 1
                    elif mod == 'RTDOSE':
                        # Optional: Keep Dose/Plan in root? User asked for Dose/Plan inside main folder.
                        # Wait, user said: "dose and plan file inside main folder"
                        # This implies keeping them in patient_dir, NOT a subfolder?
                        # Or user meant "Main Folder -> Dose File" (direct file)
                        # Let's interpret "inside main folder" as "Keep as files in PatientID/"
                        # BUT user prompt: "patient folder--> CT folder, STRUCT folder, plan file, dose file"
                        # So we DO NOT move Dose/Plan to subfolders. We leave them alone.
                        counts['RTDOSE'] += 1
                        continue # Skip moving
                    elif mod == 'RTPLAN':
                        counts['RTPLAN'] += 1
                        continue # Skip moving
                    else:
                        continue # Skip unknowns
                        
                    # Move File (Rename if duplicate exists)
                    if target_folder:
                        fname = os.path.basename(full_path)
                        dst_path = os.path.join(target_folder, fname)
                        
                        # Handle duplicate filenames
                        if os.path.exists(dst_path):
                            base, ext = os.path.splitext(fname)
                            dst_path = os.path.join(target_folder, f"{base}_dup{counts[mod]}{ext}")
                            
                        shutil.move(full_path, dst_path)
                        
                except:
                    continue # Not a DICOM
                    
        # Clean up empty folders (optional, risky if other data exists)
        # We skip deleting empty source folders to be safe.
        
        # Summary
        print(f"[{i+1}] {patient_id}: Organized {counts['CT']} CTs, {counts['RTSTRUCT']} Structs. (Dose/Plan kept in root)")

    print("\nSUCCESS! Dataset organized.")

--- STARTING UNIVERSAL ORGANIZER ---
Target: d:\Thesis_Project\Data\AMCGH
Found 27 patients. Processing...
[1] 20250982new: Organized 133 CTs, 1 Structs. (Dose/Plan kept in root)
[2] 20251575new: Organized 141 CTs, 1 Structs. (Dose/Plan kept in root)
[3] 20251577new: Organized 142 CTs, 1 Structs. (Dose/Plan kept in root)
[4] 20251626new: Organized 117 CTs, 1 Structs. (Dose/Plan kept in root)
[5] 20251673new: Organized 116 CTs, 1 Structs. (Dose/Plan kept in root)
[6] 20251705new: Organized 112 CTs, 1 Structs. (Dose/Plan kept in root)
[7] 20251727new: Organized 105 CTs, 1 Structs. (Dose/Plan kept in root)
[8] 20251773: Organized 139 CTs, 1 Structs. (Dose/Plan kept in root)
[9] New 20250871: Organized 133 CTs, 1 Structs. (Dose/Plan kept in root)
[10] New 20251222: Organized 110 CTs, 1 Structs. (Dose/Plan kept in root)
[11] New 20251320: Organized 121 CTs, 1 Structs. (Dose/Plan kept in root)
[12] New 20251360: Organized 148 CTs, 1 Structs. (Dose/Plan kept in root)
[13] New 20251384: Organi

In [4]:
import os
import shutil
import pydicom
import warnings

# --- CONFIGURATION ---
# CHANGE THIS PATH for each hospital run!
# Example: '../Data/AMCGH/' OR '../Data/SQUARE/'
source_path = '../Data/AMCGH/' 

print(f"--- STARTING UNIVERSAL ORGANIZER (CLEANUP MODE) ---")
print(f"Target: {os.path.abspath(source_path)}")

if not os.path.exists(source_path):
    print("Error: Source path not found!")
else:
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Found {len(patient_folders)} patients. Processing...")

    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        # 1. Create Target Folders (Only CT and Struct)
        ct_dir = os.path.join(patient_dir, 'CT')
        struct_dir = os.path.join(patient_dir, 'Struct')
        
        # We DO NOT create Dose/Plan folders since we want those files in root
        for d in [ct_dir, struct_dir]:
            if not os.path.exists(d): os.makedirs(d)
            
        counts = {'CT': 0, 'RTSTRUCT': 0}
        
        # 2. Scan & Move Files (Recursive)
        for root, dirs, files in os.walk(patient_dir):
            
            # Skip if we are inside our new organized folders
            if 'CT' in root or 'Struct' in root:
                continue
                
            for f in files:
                full_path = os.path.join(root, f)
                
                try:
                    # Force read header
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    mod = dcm.get("Modality", "Unknown").upper()
                    
                    target_folder = None
                    
                    if mod == 'CT':
                        target_folder = ct_dir
                        counts['CT'] += 1
                    elif mod == 'RTSTRUCT':
                        target_folder = struct_dir
                        counts['RTSTRUCT'] += 1
                    elif mod == 'RTDOSE' or mod == 'RTPLAN':
                        # Keep in root (do nothing)
                        continue 
                    else:
                        continue # Skip unknowns
                        
                    # Move File
                    if target_folder:
                        fname = os.path.basename(full_path)
                        dst_path = os.path.join(target_folder, fname)
                        
                        # Handle duplicate filenames if moving from subfolders
                        if os.path.exists(dst_path):
                            base, ext = os.path.splitext(fname)
                            dst_path = os.path.join(target_folder, f"{base}_dup{counts[mod]}{ext}")
                            
                        shutil.move(full_path, dst_path)
                        
                except:
                    continue # Not a DICOM
                    
        # 3. CLEANUP: Remove empty 'Dose' and 'Plan' folders if they exist from previous runs
        dose_garbage = os.path.join(patient_dir, 'Dose')
        plan_garbage = os.path.join(patient_dir, 'Plan')
        
        if os.path.exists(dose_garbage) and not os.listdir(dose_garbage):
            os.rmdir(dose_garbage)
        if os.path.exists(plan_garbage) and not os.listdir(plan_garbage):
            os.rmdir(plan_garbage)
        
        # print(f"[{i+1}] {patient_id}: Organized {counts['CT']} CTs, {counts['RTSTRUCT']} Structs.")

    print("\nSUCCESS! Dataset organized & cleaned.")

--- STARTING UNIVERSAL ORGANIZER (CLEANUP MODE) ---
Target: d:\Thesis_Project\Data\AMCGH
Found 27 patients. Processing...

SUCCESS! Dataset organized & cleaned.
